<a href="https://colab.research.google.com/github/vishalpgmp/Model-Compressor/blob/main/SvD_model_compression_and_MOE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling
---------------------------------------------------------------------------
This script performs a one-shot, non-trainable downsizing of a Hugging Face
Transformer model to a target active parameter budget. It uses SVD-based
latent bottleneck compression on weight matrices and optionally converts the
MLP layers into a Mixture of Experts (MoE) architecture.

GPU Memory Management:
- Processes weights one layer at a time to minimize GPU memory usage
- Moves processed weights to CPU RAM immediately after processing
- Uses memory-mapped loading for large models
- Clears GPU cache after each layer
- Optionally saves intermediate results to disk for very large models
"""

import torch
import torch.nn as nn
import math
import json
import os
import gc
import shutil
import tempfile
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import save_file, load_file
import psutil
import time

# ==========================================
# USER CONTROL & PARAMETER TWEAKING PANEL
# ==========================================
TARGET_ACTIVE_PARAMS = 5.0e8           # Target 500M active parameters
ALLOW_EMBEDDING_DOWNSIZING = True      # If True, downscales token embedding dim via SVD
DOWNSIZE_ATTENTION = True              # Keep Attention weights intact, or squeeze them
DOWNSIZE_MLP_ONLY = False              # Concentrate all SVD latent squeezing strictly on MLP/FFN layers

# MoE Architectural Upcycling Settings
CONVERT_TO_MOE = True                  # Turn the squeezed MLP layer into an MoE layout
NUM_TOTAL_EXPERTS = 8                  # Total number of generated expert clones
NUM_ACTIVE_EXPERTS = 2                 # Top-K active routing choice per token

# Memory Management Settings
USE_GPU = True                         # Use GPU if available
GPU_MEMORY_FRACTION = 0.8              # Maximum fraction of GPU memory to use
SAVE_INTERMEDIATE_TO_DISK = True       # Save intermediate results to disk to save RAM
TEMP_DIR = "./temp_processing"         # Directory for temporary files
CLEAR_GPU_CACHE_EVERY_N_LAYERS = 8     # Clear GPU cache every N layers

# Model source (can be any HF model name or local path)
MODEL_ID = "openbmb/MiniCPM5-1B-SFT"  # Replace with your model

# Output directory for the modified model
OUTPUT_DIR = "./downsized_moe_model"
# ==========================================


class MemoryManager:
    """Manages GPU/CPU memory during model processing."""

    def __init__(self, use_gpu: bool = True, gpu_memory_fraction: float = 0.8):
        self.use_gpu = use_gpu and torch.cuda.is_available()
        self.gpu_memory_fraction = gpu_memory_fraction
        self.device = torch.device("cuda" if self.use_gpu else "cpu")

        if self.use_gpu:
            # Set GPU memory fraction
            torch.cuda.set_per_process_memory_fraction(gpu_memory_fraction)
            self.gpu_name = torch.cuda.get_device_name(0)
            self.total_gpu_memory = torch.cuda.get_device_properties(0).total_memory
            print(f"Using GPU: {self.gpu_name}")
            print(f"Total GPU memory: {self.total_gpu_memory / 1024**3:.2f} GB")
            print(f"Memory fraction: {gpu_memory_fraction}")
        else:
            print("Using CPU for processing")

    def get_memory_info(self) -> Dict[str, float]:
        """Get current memory usage."""
        info = {}

        if self.use_gpu:
            info['gpu_allocated'] = torch.cuda.memory_allocated() / 1024**3
            info['gpu_cached'] = torch.cuda.memory_reserved() / 1024**3
            info['gpu_free'] = (self.total_gpu_memory - torch.cuda.memory_allocated()) / 1024**3

        # CPU memory
        cpu_memory = psutil.virtual_memory()
        info['cpu_used'] = cpu_memory.used / 1024**3
        info['cpu_available'] = cpu_memory.available / 1024**3
        info['cpu_percent'] = cpu_memory.percent

        return info

    def clear_gpu_cache(self):
        """Clear GPU cache."""
        if self.use_gpu:
            torch.cuda.empty_cache()
            gc.collect()

    def move_to_device(self, tensor: torch.Tensor, device: Optional[str] = None) -> torch.Tensor:
        """Move tensor to target device, checking memory."""
        target_device = device or self.device

        if target_device == "cuda" and self.use_gpu:
            # Check if enough GPU memory
            tensor_size_gb = tensor.numel() * tensor.element_size() / 1024**3
            free_gpu_gb = (self.total_gpu_memory - torch.cuda.memory_allocated()) / 1024**3

            if tensor_size_gb > free_gpu_gb * 0.9:  # Leave 10% buffer
                print(f"WARNING: Tensor size ({tensor_size_gb:.2f} GB) exceeds available GPU memory ({free_gpu_gb:.2f} GB)")
                print("Processing on CPU instead")
                return tensor.to("cpu")

        return tensor.to(target_device)

    def process_svd_on_optimal_device(self, W: torch.Tensor, target_rank: int):
        """Process SVD on GPU if beneficial, otherwise CPU."""
        matrix_size_gb = W.numel() * W.element_size() / 1024**3

        if self.use_gpu:
            free_gpu_gb = (self.total_gpu_memory - torch.cuda.memory_allocated()) / 1024**3
            # SVD requires ~2-3x matrix size for computation
            required_gpu_gb = matrix_size_gb * 3

            if required_gpu_gb < free_gpu_gb * 0.8:
                # Process on GPU
                W_gpu = W.to("cuda")
                U, S, V = torch.linalg.svd(W_gpu, full_matrices=False)
                del W_gpu
                self.clear_gpu_cache()

                # Move results to CPU
                U_cpu = U.cpu()
                S_cpu = S.cpu()
                V_cpu = V.cpu()
                del U, S, V
                self.clear_gpu_cache()

                return U_cpu, S_cpu, V_cpu

        # Process on CPU
        U, S, V = torch.linalg.svd(W.float().cpu(), full_matrices=False)
        return U, S, V

    def check_ram_availability(self, required_gb: float) -> bool:
        """Check if enough RAM is available."""
        available_gb = psutil.virtual_memory().available / 1024**3
        return available_gb > required_gb


class DiskStorage:
    """Handles temporary storage of intermediate results on disk."""

    def __init__(self, temp_dir: str = "./temp_processing"):
        self.temp_dir = temp_dir
        os.makedirs(temp_dir, exist_ok=True)
        self.file_counter = 0

    def save_tensor(self, tensor: torch.Tensor, name: str) -> str:
        """Save tensor to disk and return filename."""
        filename = os.path.join(self.temp_dir, f"{name}.pt")
        torch.save(tensor, filename)
        return filename

    def load_tensor(self, filename: str) -> torch.Tensor:
        """Load tensor from disk."""
        return torch.load(filename, map_location='cpu')

    def delete_file(self, filename: str):
        """Delete temporary file."""
        if os.path.exists(filename):
            os.remove(filename)

    def cleanup(self):
        """Remove all temporary files."""
        if os.path.exists(self.temp_dir):
            shutil.rmtree(self.temp_dir)


def get_tensor_size(tensor: torch.Tensor) -> int:
    """Return number of elements in a tensor."""
    return tensor.numel()


def svd_compress(
    W: torch.Tensor,
    target_rank: int,
    memory_manager: Optional[MemoryManager] = None,
    return_scale_factor: bool = True,
) -> Tuple[torch.Tensor, Optional[float]]:
    """
    Compress a weight matrix W using SVD latent bottleneck.
    Memory-optimized version that processes on optimal device.
    """
    # Clamp target rank
    target_rank = max(1, min(target_rank, W.shape[0]))

    if memory_manager and memory_manager.use_gpu:
        # Use memory manager for optimal device processing
        U, S, V = memory_manager.process_svd_on_optimal_device(W, target_rank)
    else:
        # CPU processing
        U, S, V = torch.linalg.svd(W.float().cpu(), full_matrices=False)

    # Top-k singular values
    S_top = S[:target_rank]

    # Reconstruct compressed matrix
    W_compressed = (U[:, :target_rank] * S_top.unsqueeze(0)) @ V[:target_rank, :]

    scale_factor = None
    if return_scale_factor:
        # Scale correction to preserve total operational energy
        total_singular = torch.sum(S)
        if total_singular > 1e-12:
            scale_factor = (total_singular / torch.sum(S_top)).item()
        else:
            scale_factor = 1.0
        W_compressed = W_compressed * scale_factor

    # Clean up
    del U, S, V, S_top
    if memory_manager:
        memory_manager.clear_gpu_cache()

    return W_compressed.to(W.dtype), scale_factor


def calculate_param_count(state_dict: Dict[str, torch.Tensor]) -> int:
    """Total number of parameters in a state dict."""
    return sum(t.numel() for t in state_dict.values())


def calculate_active_params(state_dict: Dict[str, torch.Tensor], num_active_experts: int) -> int:
    """Calculate active parameters for MoE model."""
    active_params = 0
    for key, tensor in state_dict.items():
        if "expert" in key and ("gate_experts" in key or "up_experts" in key or "down_experts" in key):
            # Only count active experts
            active_params += num_active_experts * tensor[0].numel()
        else:
            active_params += tensor.numel()
    return active_params


def analyze_model(config: AutoConfig) -> Dict[str, Any]:
    """Analyze model dimensions and parameter counts."""
    hidden_size = config.hidden_size
    vocab_size = config.vocab_size
    num_layers = config.num_hidden_layers
    intermediate_size = config.intermediate_size
    num_attention_heads = config.num_attention_heads
    head_dim = hidden_size // num_attention_heads

    # Parameter counts
    embedding_params = vocab_size * hidden_size
    lm_head_params = vocab_size * hidden_size

    # Attention params per layer
    attn_params_per_layer = 4 * hidden_size * hidden_size

    # MLP params per layer
    mlp_params_per_layer = 3 * hidden_size * intermediate_size

    total_attention = num_layers * attn_params_per_layer
    total_mlp = num_layers * mlp_params_per_layer
    total = embedding_params + lm_head_params + total_attention + total_mlp

    return {
        "hidden_size": hidden_size,
        "vocab_size": vocab_size,
        "num_layers": num_layers,
        "intermediate_size": intermediate_size,
        "num_attention_heads": num_attention_heads,
        "head_dim": head_dim,
        "embedding_params": embedding_params,
        "lm_head_params": lm_head_params,
        "attention_params": total_attention,
        "mlp_params": total_mlp,
        "total_params": total,
        "attn_params_per_layer": attn_params_per_layer,
        "mlp_params_per_layer": mlp_params_per_layer,
    }


def modify_config_for_downsizing(
    config: AutoConfig,
    new_hidden_size: Optional[int] = None,
    new_intermediate_size: Optional[int] = None,
) -> AutoConfig:
    """Update config to reflect modifications."""
    if new_hidden_size is not None:
        config.hidden_size = new_hidden_size
        if hasattr(config, 'num_attention_heads'):
            num_heads = config.num_attention_heads
            if new_hidden_size % num_heads != 0:
                new_num_heads = math.gcd(new_hidden_size, num_heads)
                while new_hidden_size % new_num_heads != 0:
                    new_num_heads -= 1
                config.num_attention_heads = new_num_heads
                print(f"Adjusted num_attention_heads to {new_num_heads}")
            config.head_dim = new_hidden_size // config.num_attention_heads
        else:
            config.head_dim = new_hidden_size // config.num_attention_heads

    if new_intermediate_size is not None:
        config.intermediate_size = new_intermediate_size
    return config


def build_moe_parameters(
    original_gate_proj: torch.Tensor,
    original_up_proj: torch.Tensor,
    original_down_proj: torch.Tensor,
    hidden_size: int,
    expert_size: int,
    num_total_experts: int,
    num_active_experts: int,
    dtype: torch.dtype,
    memory_manager: MemoryManager,
) -> Dict[str, torch.Tensor]:
    """
    Build MoE parameters from original MLP weights.
    Memory-optimized version.
    """
    print(f"  Compressing gate_proj...")
    gate_compressed, _ = svd_compress(original_gate_proj, expert_size, memory_manager)
    print(f"  Compressing up_proj...")
    up_compressed, _ = svd_compress(original_up_proj, expert_size, memory_manager)
    print(f"  Compressing down_proj...")
    down_compressed, _ = svd_compress(original_down_proj, expert_size, memory_manager)

    # Create expert stacks on CPU to save GPU memory
    gate_experts = gate_compressed.unsqueeze(0).expand(num_total_experts, -1, -1).clone().cpu()
    up_experts = up_compressed.unsqueeze(0).expand(num_total_experts, -1, -1).clone().cpu()
    down_experts = down_compressed.unsqueeze(0).expand(num_total_experts, -1, -1).clone().cpu()

    # Router parameters
    router_weight = torch.empty(hidden_size, num_total_experts, dtype=dtype)
    nn.init.uniform_(router_weight, -0.05, 0.05)
    router_weight[:, 0] += 0.1

    # Clean up intermediate tensors
    del gate_compressed, up_compressed, down_compressed
    memory_manager.clear_gpu_cache()

    return {
        "gate_experts": gate_experts,
        "up_experts": up_experts,
        "down_experts": down_experts,
        "router_weight": router_weight,
        "num_total_experts": num_total_experts,
        "num_active_experts": num_active_experts,
        "expert_size": expert_size,
    }


def calculate_optimal_budget(
    analysis: Dict[str, Any],
    target_params: int,
    allow_embedding_downsize: bool,
    downsample_attention: bool,
) -> Dict[str, Any]:
    """Calculate optimal budget allocation."""
    hidden_size = analysis['hidden_size']
    num_layers = analysis['num_layers']

    total_params = analysis['total_params']
    reduction_ratio = target_params / total_params

    print(f"\n--- Budget Allocation ---")
    print(f"Total original params: {total_params:,}")
    print(f"Target params: {target_params:,}")
    print(f"Required reduction ratio: {reduction_ratio:.4f}")

    fixed_params = analysis['embedding_params'] + analysis['lm_head_params']

    if fixed_params > target_params and not allow_embedding_downsize:
        print(f"ERROR: Embedding + LM Head ({fixed_params:,}) exceeds target ({target_params:,})")
        raise ValueError("Cannot meet target without embedding downsizing")

    new_hidden_size = hidden_size
    if allow_embedding_downsize and fixed_params > target_params * 0.5:
        remaining_for_embedding = target_params * 0.4
        vocab_size = analysis['vocab_size']
        new_hidden_size = int(remaining_for_embedding / (2 * vocab_size))
        new_hidden_size = max(64, min(new_hidden_size, hidden_size))
        print(f"Reducing embedding hidden size from {hidden_size} to {new_hidden_size}")

    embedding_lm_head_params = analysis['vocab_size'] * new_hidden_size * 2
    remaining_budget = target_params - embedding_lm_head_params

    print(f"Embedding + LM Head params: {embedding_lm_head_params:,}")
    print(f"Remaining budget for transformer: {remaining_budget:,}")

    if downsample_attention:
        attention_budget = remaining_budget * 0.4
        mlp_budget = remaining_budget * 0.6
    else:
        attention_budget = analysis['attention_params']
        mlp_budget = remaining_budget - attention_budget

        if mlp_budget < 0:
            print(f"WARNING: Attention alone exceeds remaining budget")
            downsample_attention = True
            attention_budget = remaining_budget * 0.4
            mlp_budget = remaining_budget * 0.6

    attn_params_per_layer = analysis['attn_params_per_layer']
    attn_reduction = attention_budget / analysis['attention_params'] if analysis['attention_params'] > 0 else 1.0
    new_attn_rank = max(1, int(hidden_size * math.sqrt(max(attn_reduction, 0.01))))

    mlp_params_per_layer = analysis['mlp_params_per_layer']
    if mlp_budget > 0:
        new_intermediate_size = int(mlp_budget / (3 * hidden_size * num_layers))
    else:
        new_intermediate_size = 1

    new_intermediate_size = max(8, min(new_intermediate_size, analysis['intermediate_size']))
    new_attn_rank = max(8, min(new_attn_rank, hidden_size))

    print(f"Attention budget: {attention_budget:,} (compression: {attn_reduction:.4f})")
    print(f"New attention rank: {new_attn_rank}")
    print(f"MLP budget: {mlp_budget:,}")
    print(f"New intermediate size: {new_intermediate_size}")

    return {
        "new_hidden_size": new_hidden_size,
        "new_intermediate_size": new_intermediate_size,
        "new_attn_rank": new_attn_rank,
        "downsample_attention": downsample_attention,
        "attention_budget": attention_budget,
        "mlp_budget": mlp_budget,
    }


def load_model_layer_by_layer(model_id: str):
    """Load model weights layer by layer using memory mapping."""
    from safetensors import safe_open

    # Find safetensors files
    index_file = os.path.join(model_id, "model.safetensors.index.json")
    if os.path.exists(index_file):
        with open(index_file, 'r') as f:
            index = json.load(f)
        weight_files = list(set(index['weight_map'].values()))
    else:
        # Single file
        weight_files = ["model.safetensors"]
        if not os.path.exists(os.path.join(model_id, "model.safetensors")):
            # Try to download
            print(f"Downloading model files...")
            model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)
            model.save_pretrained("./temp_model_files")
            model_id = "./temp_model_files"
            weight_files = ["model.safetensors"]

    return model_id, weight_files


def main():
    print("=" * 80)
    print("Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling")
    print("=" * 80)

    # Initialize memory manager
    memory_manager = MemoryManager(use_gpu=USE_GPU, gpu_memory_fraction=GPU_MEMORY_FRACTION)

    # Initialize disk storage for intermediate results
    disk_storage = DiskStorage(TEMP_DIR) if SAVE_INTERMEDIATE_TO_DISK else None

    # Load model config
    print(f"\nLoading configuration from: {MODEL_ID}")
    config = AutoConfig.from_pretrained(MODEL_ID)

    # Analyze model
    analysis = analyze_model(config)
    print("\n--- Model Analysis ---")
    print(f"Hidden size: {analysis['hidden_size']}")
    print(f"Vocab size: {analysis['vocab_size']}")
    print(f"Num layers: {analysis['num_layers']}")
    print(f"Intermediate size: {analysis['intermediate_size']}")
    print(f"Embedding params: {analysis['embedding_params']:,}")
    print(f"LM Head params: {analysis['lm_head_params']:,}")
    print(f"Attention params: {analysis['attention_params']:,}")
    print(f"MLP params: {analysis['mlp_params']:,}")
    print(f"Total params: {analysis['total_params']:,}")
    print(f"Target active params: {TARGET_ACTIVE_PARAMS:,}")

    # Calculate optimal budget allocation
    budget = calculate_optimal_budget(
        analysis,
        TARGET_ACTIVE_PARAMS,
        ALLOW_EMBEDDING_DOWNSIZING,
        DOWNSIZE_ATTENTION or DOWNSIZE_MLP_ONLY == False,
    )

    new_hidden_size = budget['new_hidden_size']
    new_intermediate_size = budget['new_intermediate_size']
    new_attn_rank = budget['new_attn_rank']
    downsample_attention = budget['downsample_attention']

    # Load model weights with memory optimization
    print(f"\nLoading model weights from: {MODEL_ID}")
    print("Using memory-efficient loading...")

    # Load model in a memory-efficient way
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
        device_map="cpu",  # Load on CPU first
        low_cpu_mem_usage=True,
    )
    state_dict = model.state_dict()

    # Free model object to save memory
    del model
    memory_manager.clear_gpu_cache()

    # Create new state dict for modified model (stored on CPU)
    new_state_dict = {}

    # Process embedding and lm_head
    print("\n--- Step 2: SVD Latent Bottleneck Compression ---")

    embed_key = "model.embed_tokens.weight"
    lm_head_key = "lm_head.weight"

    if new_hidden_size != analysis['hidden_size']:
        # Compress embedding
        print("Compressing embedding...")
        embed_weight = state_dict[embed_key]
        embed_compressed, _ = svd_compress(embed_weight.T, new_hidden_size, memory_manager)
        new_state_dict[embed_key] = embed_compressed.T

        # Add projection layers
        proj_up = torch.randn(new_hidden_size, analysis['hidden_size']) * 0.02
        new_state_dict["model.embed_proj_up.weight"] = proj_up
        new_state_dict["model.embed_proj_up.bias"] = torch.zeros(analysis['hidden_size'])

        # Compress lm_head
        print("Compressing lm_head...")
        if lm_head_key in state_dict:
            lm_head_weight = state_dict[lm_head_key]
            lm_head_compressed, _ = svd_compress(lm_head_weight.T, new_hidden_size, memory_manager)
            new_state_dict[lm_head_key] = lm_head_compressed.T
        else:
            new_state_dict[lm_head_key] = new_state_dict[embed_key].clone()

        # Add projection before lm_head
        proj_down = torch.randn(analysis['hidden_size'], new_hidden_size) * 0.02
        new_state_dict["model.lm_head_proj_down.weight"] = proj_down
        new_state_dict["model.lm_head_proj_down.bias"] = torch.zeros(new_hidden_size)

        print(f"Embedding compressed from {analysis['hidden_size']} to {new_hidden_size}")

        # Free memory
        del embed_weight, embed_compressed, lm_head_weight
        if 'lm_head_compressed' in locals():
            del lm_head_compressed
        memory_manager.clear_gpu_cache()
    else:
        # Keep embedding as is
        new_state_dict[embed_key] = state_dict[embed_key]
        if lm_head_key in state_dict:
            new_state_dict[lm_head_key] = state_dict[lm_head_key]
        else:
            new_state_dict[lm_head_key] = new_state_dict[embed_key].clone()

    # Process each transformer layer
    print("\nProcessing transformer layers...")
    print("Layer-by-layer processing with memory optimization:")

    for layer_idx in range(analysis['num_layers']):
        print(f"\n--- Layer {layer_idx + 1}/{analysis['num_layers']} ---")
        prefix = f"model.layers.{layer_idx}."

        # Track memory usage
        mem_info = memory_manager.get_memory_info()
        if memory_manager.use_gpu:
            print(f"  GPU Memory: {mem_info['gpu_allocated']:.2f} GB allocated, {mem_info['gpu_free']:.2f} GB free")
        print(f"  CPU Memory: {mem_info['cpu_used']:.2f} GB used, {mem_info['cpu_available']:.2f} GB available")

        # Attention layers
        attn_prefix = prefix + "self_attn."
        for proj_name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
            key = attn_prefix + proj_name + ".weight"
            if key in state_dict:
                if downsample_attention:
                    # Compress attention
                    W = state_dict[key]
                    W_compressed, _ = svd_compress(W, new_attn_rank, memory_manager)
                    new_state_dict[key] = W_compressed
                    del W, W_compressed
                else:
                    new_state_dict[key] = state_dict[key]

                # Copy bias if exists
                bias_key = attn_prefix + proj_name + ".bias"
                if bias_key in state_dict:
                    new_state_dict[bias_key] = state_dict[bias_key]

        # MLP layers
        mlp_prefix = prefix + "mlp."

        if CONVERT_TO_MOE:
            # Build MoE from original MLP
            gate_key = mlp_prefix + "gate_proj.weight"
            up_key = mlp_prefix + "up_proj.weight"
            down_key = mlp_prefix + "down_proj.weight"

            if gate_key in state_dict:
                gate_orig = state_dict[gate_key]
                up_orig = state_dict[up_key]
                down_orig = state_dict[down_key]

                # Calculate expert size
                available_per_layer = budget['mlp_budget'] / analysis['num_layers']
                router_size = analysis['hidden_size'] * NUM_TOTAL_EXPERTS
                expert_params_per_layer = available_per_layer - router_size

                if expert_params_per_layer > 0:
                    expert_size = int(expert_params_per_layer / (3 * analysis['hidden_size'] * NUM_ACTIVE_EXPERTS))
                    expert_size = max(16, min(expert_size, analysis['intermediate_size']))
                else:
                    expert_size = 16

                print(f"  Creating MoE with {NUM_TOTAL_EXPERTS} experts, expert_size={expert_size}")

                moe_params = build_moe_parameters(
                    gate_orig, up_orig, down_orig,
                    analysis['hidden_size'], expert_size,
                    NUM_TOTAL_EXPERTS, NUM_ACTIVE_EXPERTS,
                    dtype=torch.float32,
                    memory_manager=memory_manager,
                )

                # Store MoE parameters
                new_state_dict[mlp_prefix + "gate_experts"] = moe_params["gate_experts"]
                new_state_dict[mlp_prefix + "up_experts"] = moe_params["up_experts"]
                new_state_dict[mlp_prefix + "down_experts"] = moe_params["down_experts"]
                new_state_dict[mlp_prefix + "router.weight"] = moe_params["router_weight"]

                # Free memory
                del gate_orig, up_orig, down_orig, moe_params

        else:
            # Regular MLP compression
            for proj_name in ["gate_proj", "up_proj", "down_proj"]:
                key = mlp_prefix + proj_name + ".weight"
                if key in state_dict:
                    W = state_dict[key]
                    W_compressed, _ = svd_compress(W, new_intermediate_size, memory_manager)
                    new_state_dict[key] = W_compressed
                    del W, W_compressed

                    # Copy bias if exists
                    bias_key = mlp_prefix + proj_name + ".bias"
                    if bias_key in state_dict:
                        new_state_dict[bias_key] = state_dict[bias_key]

        # Copy LayerNorm weights
        for norm_name in ["input_layernorm.weight", "post_attention_layernorm.weight"]:
            key = prefix + norm_name
            if key in state_dict:
                new_state_dict[key] = state_dict[key]

        # Copy bias if exists
        for bias_name in ["input_layernorm.bias", "post_attention_layernorm.bias"]:
            key = prefix + bias_name
            if key in state_dict:
                new_state_dict[key] = state_dict[bias_name]

        # Remove layer from state_dict to free memory
        keys_to_delete = [key for key in state_dict.keys() if key.startswith(prefix)]
        for key in keys_to_delete:
            del state_dict[key]

        # Clear GPU cache periodically
        if (layer_idx + 1) % CLEAR_GPU_CACHE_EVERY_N_LAYERS == 0:
            memory_manager.clear_gpu_cache()
            print(f"  Cleared GPU cache after layer {layer_idx + 1}")

        # Save intermediate results to disk if RAM usage is high
        if SAVE_INTERMEDIATE_TO_DISK and layer_idx > 0 and layer_idx % 5 == 0:
            # Check RAM usage
            cpu_percent = psutil.virtual_memory().percent
            if cpu_percent > 80:
                print(f"  High RAM usage ({cpu_percent}%). Saving intermediate results to disk...")
                intermediate_file = os.path.join(TEMP_DIR, f"layer_{layer_idx}_checkpoint.pt")
                torch.save(new_state_dict, intermediate_file)
                # Clear some tensors from new_state_dict to free RAM
                for key in list(new_state_dict.keys()):
                    if key.startswith(f"model.layers.{max(0, layer_idx - 2)}."):
                        del new_state_dict[key]

    # Copy final layernorm
    for key in state_dict:
        if "model.norm" in key:
            new_state_dict[key] = state_dict[key]

    # Free remaining state_dict
    del state_dict
    memory_manager.clear_gpu_cache()

    # Calculate actual active parameters
    print("\n--- Summary ---")
    total_new_params = calculate_param_count(new_state_dict)

    if CONVERT_TO_MOE:
        active_params = calculate_active_params(new_state_dict, NUM_ACTIVE_EXPERTS)
        print(f"Total parameters stored: {total_new_params:,}")
        print(f"Active parameters (during inference): {active_params:,}")
        print(f"Target active parameters: {TARGET_ACTIVE_PARAMS:,}")
        print(f"Compression ratio: {analysis['total_params'] / active_params:.2f}x")
    else:
        print(f"Total new parameters: {total_new_params:,}")
        print(f"Target active parameters: {TARGET_ACTIVE_PARAMS:,}")
        print(f"Compression ratio: {analysis['total_params'] / total_new_params:.2f}x")

    # Update config
    print("\nUpdating configuration...")
    updated_config = config

    if new_hidden_size != analysis['hidden_size']:
        updated_config.hidden_size = analysis['hidden_size']
        updated_config.embedding_size = new_hidden_size
        updated_config.has_embed_proj = True
    else:
        updated_config.embedding_size = analysis['hidden_size']

    if CONVERT_TO_MOE:
        updated_config.moe_num_experts = NUM_TOTAL_EXPERTS
        updated_config.moe_active_experts = NUM_ACTIVE_EXPERTS
        updated_config.moe_expert_size = expert_size
        updated_config.intermediate_size = expert_size
    else:
        updated_config.intermediate_size = new_intermediate_size

    # Save model
    print(f"\nSaving model to: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Save state dict as safetensors
    print("Saving weights...")
    save_file(new_state_dict, os.path.join(OUTPUT_DIR, "model.safetensors"))

    # Save config
    print("Saving config...")
    updated_config.save_pretrained(OUTPUT_DIR)

    # Save tokenizer if available
    try:
        print("Saving tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print("Tokenizer saved.")
    except:
        print("Could not save tokenizer.")

    # Cleanup temporary files
    if disk_storage:
        print("Cleaning up temporary files...")
        disk_storage.cleanup()

    # Final memory report
    print("\n--- Final Memory Report ---")
    mem_info = memory_manager.get_memory_info()
    if memory_manager.use_gpu:
        print(f"GPU Memory: {mem_info['gpu_allocated']:.2f} GB allocated")
    print(f"CPU Memory: {mem_info['cpu_used']:.2f} GB used")

    print("\n✅ Model surgery complete! Output saved to:", OUTPUT_DIR)
    print("=" * 80)


if __name__ == "__main__":
    main()

Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling
Using GPU: Tesla T4
Total GPU memory: 14.56 GB
Memory fraction: 0.8

Loading configuration from: openbmb/MiniCPM5-1B-SFT



--- Model Analysis ---
Hidden size: 1536
Vocab size: 130560
Num layers: 24
Intermediate size: 4608
Embedding params: 200,540,160
LM Head params: 200,540,160
Attention params: 226,492,416
MLP params: 509,607,936
Total params: 1,137,180,672
Target active params: 500,000,000.0

--- Budget Allocation ---
Total original params: 1,137,180,672
Target params: 500,000,000.0
Required reduction ratio: 0.4397
Reducing embedding hidden size from 1536 to 765
Embedding + LM Head params: 199,756,800
Remaining budget for transformer: 300,243,200.0
Attention budget: 120,097,280.0 (compression: 0.5302)
New attention rank: 1118
MLP budget: 180,145,920.0
New intermediate size: 1628

Loading model weights from: openbmb/MiniCPM5-1B-SFT
Using memory-efficient loading...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]


--- Step 2: SVD Latent Bottleneck Compression ---
Compressing embedding...
Compressing lm_head...
Embedding compressed from 1536 to 765

Processing transformer layers...
Layer-by-layer processing with memory optimization:

--- Layer 1/24 ---
  GPU Memory: 0.00 GB allocated, 14.56 GB free
  CPU Memory: 8.57 GB used, 3.77 GB available
  Creating MoE with 8 experts, expert_size=813
  Compressing gate_proj...
  Compressing up_proj...
  Compressing down_proj...

--- Layer 2/24 ---
  GPU Memory: 0.00 GB allocated, 14.56 GB free
  CPU Memory: 9.17 GB used, 3.18 GB available
  Creating MoE with 8 experts, expert_size=813
  Compressing gate_proj...
  Compressing up_proj...
  Compressing down_proj...

--- Layer 3/24 ---
  GPU Memory: 0.00 GB allocated, 14.56 GB free
  CPU Memory: 9.75 GB used, 2.60 GB available
  Creating MoE with 8 experts, expert_size=813
  Compressing gate_proj...
  Compressing up_proj...
  Compressing down_proj...

--- Layer 4/24 ---
  GPU Memory: 0.00 GB allocated, 14.56 G

In [ ]:
"""
Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling
---------------------------------------------------------------------------
Memory-optimized version that processes layers one at a time and saves
directly to disk to prevent OOM errors.
"""

import torch
import torch.nn as nn
import math
import json
import os
import gc
import shutil
import tempfile
import time
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import save_file, load_file
from safetensors import safe_open
import psutil
from collections import OrderedDict

# ==========================================
# USER CONTROL & PARAMETER TWEAKING PANEL
# ==========================================
TARGET_ACTIVE_PARAMS = 5.0e8           # Target 500M active parameters
ALLOW_EMBEDDING_DOWNSIZING = True      # If True, downscales token embedding dim via SVD
DOWNSIZE_ATTENTION = True              # Keep Attention weights intact, or squeeze them
DOWNSIZE_MLP_ONLY = False              # Concentrate all SVD latent squeezing strictly on MLP/FFN layers

# MoE Architectural Upcycling Settings
CONVERT_TO_MOE = True                  # Turn the squeezed MLP layer into an MoE layout
NUM_TOTAL_EXPERTS = 4                  # Reduced from 8 to save memory
NUM_ACTIVE_EXPERTS = 2                 # Top-K active routing choice per token

# Memory Management Settings
USE_GPU = True                         # Use GPU if available
GPU_MEMORY_FRACTION = 0.8              # Maximum fraction of GPU memory to use
SAVE_LAYERS_TO_DISK = True             # Save each layer to disk immediately
TEMP_DIR = "./temp_layer_storage"      # Directory for temporary layer files
MAX_CPU_MEMORY_PERCENT = 70            # Maximum CPU memory usage before forcing disk save

# Model source
MODEL_ID = "openbmb/MiniCPM5-1B-SFT"

# Output directory
OUTPUT_DIR = "./downsized_moe_model"
# ==========================================


class MemoryManager:
    """Manages GPU/CPU memory during model processing."""

    def __init__(self, use_gpu: bool = True, gpu_memory_fraction: float = 0.8):
        self.use_gpu = use_gpu and torch.cuda.is_available()
        self.gpu_memory_fraction = gpu_memory_fraction
        self.device = torch.device("cuda" if self.use_gpu else "cpu")

        if self.use_gpu:
            torch.cuda.set_per_process_memory_fraction(gpu_memory_fraction)
            self.gpu_name = torch.cuda.get_device_name(0)
            self.total_gpu_memory = torch.cuda.get_device_properties(0).total_memory
            print(f"Using GPU: {self.gpu_name}")
            print(f"Total GPU memory: {self.total_gpu_memory / 1024**3:.2f} GB")

    def get_memory_info(self) -> Dict[str, float]:
        """Get current memory usage."""
        info = {}
        if self.use_gpu:
            info['gpu_allocated'] = torch.cuda.memory_allocated() / 1024**3
            info['gpu_cached'] = torch.cuda.memory_reserved() / 1024**3
            info['gpu_free'] = (self.total_gpu_memory - torch.cuda.memory_allocated()) / 1024**3

        cpu_memory = psutil.virtual_memory()
        info['cpu_used'] = cpu_memory.used / 1024**3
        info['cpu_available'] = cpu_memory.available / 1024**3
        info['cpu_percent'] = cpu_memory.percent

        return info

    def clear_gpu_cache(self):
        """Clear GPU cache."""
        if self.use_gpu:
            torch.cuda.empty_cache()
            gc.collect()

    def should_use_gpu(self, tensor_size_gb: float) -> bool:
        """Check if tensor fits in GPU memory."""
        if not self.use_gpu:
            return False
        free_gpu_gb = (self.total_gpu_memory - torch.cuda.memory_allocated()) / 1024**3
        return tensor_size_gb * 3 < free_gpu_gb * 0.8  # SVD needs ~3x memory


class LayerStorage:
    """Manages saving/loading layers to disk to save RAM."""

    def __init__(self, temp_dir: str = "./temp_layer_storage"):
        self.temp_dir = temp_dir
        os.makedirs(temp_dir, exist_ok=True)
        self.layer_files = {}

    def save_layer(self, layer_idx: int, layer_state: Dict[str, torch.Tensor]):
        """Save a layer's state dict to disk."""
        filename = os.path.join(self.temp_dir, f"layer_{layer_idx:03d}.pt")
        torch.save(layer_state, filename)
        self.layer_files[layer_idx] = filename
        return filename

    def load_layer(self, layer_idx: int) -> Dict[str, torch.Tensor]:
        """Load a layer's state dict from disk."""
        filename = self.layer_files.get(layer_idx)
        if filename and os.path.exists(filename):
            return torch.load(filename, map_location='cpu')
        return None

    def cleanup_layer(self, layer_idx: int):
        """Remove a layer's file from disk."""
        filename = self.layer_files.get(layer_idx)
        if filename and os.path.exists(filename):
            os.remove(filename)
            del self.layer_files[layer_idx]

    def cleanup_all(self):
        """Remove all temporary files."""
        if os.path.exists(self.temp_dir):
            shutil.rmtree(self.temp_dir)


def svd_compress_cpu(W: torch.Tensor, target_rank: int, return_scale_factor: bool = True):
    """CPU-only SVD compression to avoid GPU memory issues."""
    W_cpu = W.float().cpu()

    # For very large matrices, use randomized SVD to save memory
    if W_cpu.shape[0] * W_cpu.shape[1] > 10_000_000:  # >10M elements
        U, S, V = randomized_svd(W_cpu, target_rank)
    else:
        U, S, V = torch.linalg.svd(W_cpu, full_matrices=False)

    target_rank = max(1, min(target_rank, S.shape[0]))
    S_top = S[:target_rank]

    # Reconstruct
    W_compressed = (U[:, :target_rank] * S_top.unsqueeze(0)) @ V[:target_rank, :]

    scale_factor = None
    if return_scale_factor:
        total_singular = torch.sum(S)
        if total_singular > 1e-12:
            scale_factor = (total_singular / torch.sum(S_top)).item()
        else:
            scale_factor = 1.0
        W_compressed = W_compressed * scale_factor

    del U, S, V, S_top, W_cpu
    gc.collect()

    return W_compressed.to(W.dtype), scale_factor


def randomized_svd(W: torch.Tensor, rank: int, n_oversamples: int = 10):
    """Randomized SVD for large matrices to save memory."""
    m, n = W.shape
    rank = min(rank, m, n)

    # Generate random matrix for sketching
    Omega = torch.randn(n, rank + n_oversamples, device=W.device)

    # Compute sketch
    Y = W @ Omega
    Q, _ = torch.linalg.qr(Y)

    # Project to smaller space
    B = Q.T @ W

    # SVD on smaller matrix
    U_tilde, S, V = torch.linalg.svd(B, full_matrices=False)
    U = Q @ U_tilde

    del Omega, Y, Q, B, U_tilde
    gc.collect()

    return U[:, :rank], S[:rank], V[:rank, :]


def analyze_model(config: AutoConfig) -> Dict[str, Any]:
    """Analyze model dimensions and parameter counts."""
    hidden_size = config.hidden_size
    vocab_size = config.vocab_size
    num_layers = config.num_hidden_layers
    intermediate_size = config.intermediate_size

    embedding_params = vocab_size * hidden_size
    lm_head_params = vocab_size * hidden_size
    attn_params_per_layer = 4 * hidden_size * hidden_size
    mlp_params_per_layer = 3 * hidden_size * intermediate_size

    total_attention = num_layers * attn_params_per_layer
    total_mlp = num_layers * mlp_params_per_layer
    total = embedding_params + lm_head_params + total_attention + total_mlp

    return {
        "hidden_size": hidden_size,
        "vocab_size": vocab_size,
        "num_layers": num_layers,
        "intermediate_size": intermediate_size,
        "embedding_params": embedding_params,
        "lm_head_params": lm_head_params,
        "attention_params": total_attention,
        "mlp_params": total_mlp,
        "total_params": total,
        "attn_params_per_layer": attn_params_per_layer,
        "mlp_params_per_layer": mlp_params_per_layer,
    }


def calculate_optimal_budget(analysis, target_params, allow_embedding_downsize, downsample_attention):
    """Calculate optimal budget allocation."""
    hidden_size = analysis['hidden_size']
    num_layers = analysis['num_layers']
    total_params = analysis['total_params']
    reduction_ratio = target_params / total_params

    print(f"\n--- Budget Allocation ---")
    print(f"Required reduction ratio: {reduction_ratio:.4f}")

    fixed_params = analysis['embedding_params'] + analysis['lm_head_params']

    new_hidden_size = hidden_size
    if allow_embedding_downsize and fixed_params > target_params * 0.5:
        remaining_for_embedding = target_params * 0.4
        vocab_size = analysis['vocab_size']
        new_hidden_size = int(remaining_for_embedding / (2 * vocab_size))
        new_hidden_size = max(64, min(new_hidden_size, hidden_size))
        print(f"Reducing embedding hidden size to {new_hidden_size}")

    embedding_lm_head_params = analysis['vocab_size'] * new_hidden_size * 2
    remaining_budget = target_params - embedding_lm_head_params

    if downsample_attention:
        attention_budget = remaining_budget * 0.4
        mlp_budget = remaining_budget * 0.6
    else:
        attention_budget = analysis['attention_params']
        mlp_budget = remaining_budget - attention_budget
        if mlp_budget < 0:
            downsample_attention = True
            attention_budget = remaining_budget * 0.4
            mlp_budget = remaining_budget * 0.6

    attn_reduction = attention_budget / analysis['attention_params'] if analysis['attention_params'] > 0 else 1.0
    new_attn_rank = max(1, int(hidden_size * math.sqrt(max(attn_reduction, 0.01))))

    if mlp_budget > 0:
        new_intermediate_size = int(mlp_budget / (3 * hidden_size * num_layers))
    else:
        new_intermediate_size = 1

    new_intermediate_size = max(8, min(new_intermediate_size, analysis['intermediate_size']))
    new_attn_rank = max(8, min(new_attn_rank, hidden_size))

    print(f"Embedding + LM Head params: {embedding_lm_head_params:,}")
    print(f"Remaining budget: {remaining_budget:,}")
    print(f"Attention budget: {attention_budget:,}")
    print(f"MLP budget: {mlp_budget:,}")
    print(f"New attention rank: {new_attn_rank}")
    print(f"New intermediate size: {new_intermediate_size}")

    return {
        "new_hidden_size": new_hidden_size,
        "new_intermediate_size": new_intermediate_size,
        "new_attn_rank": new_attn_rank,
        "downsample_attention": downsample_attention,
        "attention_budget": attention_budget,
        "mlp_budget": mlp_budget,
    }


def process_embedding(state_dict, new_hidden_size, analysis, memory_manager):
    """Process embedding and lm_head with memory optimization."""
    print("\n--- Processing Embedding & LM Head ---")

    embed_key = "model.embed_tokens.weight"
    lm_head_key = "lm_head.weight"
    new_state = {}

    if new_hidden_size != analysis['hidden_size']:
        print("Compressing embedding...")
        embed_weight = state_dict[embed_key]
        embed_compressed, _ = svd_compress_cpu(embed_weight.T, new_hidden_size)
        new_state[embed_key] = embed_compressed.T

        # Add projection layers
        proj_up = torch.randn(new_hidden_size, analysis['hidden_size']) * 0.02
        new_state["model.embed_proj_up.weight"] = proj_up
        new_state["model.embed_proj_up.bias"] = torch.zeros(analysis['hidden_size'])

        print("Compressing lm_head...")
        if lm_head_key in state_dict:
            lm_head_weight = state_dict[lm_head_key]
            lm_head_compressed, _ = svd_compress_cpu(lm_head_weight.T, new_hidden_size)
            new_state[lm_head_key] = lm_head_compressed.T
        else:
            new_state[lm_head_key] = new_state[embed_key].clone()

        proj_down = torch.randn(analysis['hidden_size'], new_hidden_size) * 0.02
        new_state["model.lm_head_proj_down.weight"] = proj_down
        new_state["model.lm_head_proj_down.bias"] = torch.zeros(new_hidden_size)

        del embed_weight, embed_compressed
        if 'lm_head_weight' in locals():
            del lm_head_weight
        if 'lm_head_compressed' in locals():
            del lm_head_compressed
        gc.collect()
    else:
        new_state[embed_key] = state_dict[embed_key]
        if lm_head_key in state_dict:
            new_state[lm_head_key] = state_dict[lm_head_key]
        else:
            new_state[lm_head_key] = new_state[embed_key].clone()

    return new_state


def process_single_layer(layer_idx, layer_state, analysis, budget, memory_manager):
    """Process a single transformer layer."""
    prefix = f"model.layers.{layer_idx}."
    new_layer_state = {}

    hidden_size = analysis['hidden_size']
    new_attn_rank = budget['new_attn_rank']

    # Process attention
    attn_prefix = prefix + "self_attn."
    for proj_name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        key = attn_prefix + proj_name + ".weight"
        if key in layer_state:
            if budget['downsample_attention']:
                W = layer_state[key]
                W_compressed, _ = svd_compress_cpu(W, new_attn_rank)
                new_layer_state[key] = W_compressed
                del W, W_compressed
            else:
                new_layer_state[key] = layer_state[key]

            bias_key = attn_prefix + proj_name + ".bias"
            if bias_key in layer_state:
                new_layer_state[bias_key] = layer_state[bias_key]

    # Process MLP
    mlp_prefix = prefix + "mlp."

    if CONVERT_TO_MOE:
        gate_key = mlp_prefix + "gate_proj.weight"
        up_key = mlp_prefix + "up_proj.weight"
        down_key = mlp_prefix + "down_proj.weight"

        if gate_key in layer_state:
            gate_orig = layer_state[gate_key]
            up_orig = layer_state[up_key]
            down_orig = layer_state[down_key]

            # Calculate expert size
            available_per_layer = budget['mlp_budget'] / analysis['num_layers']
            router_size = hidden_size * NUM_TOTAL_EXPERTS
            expert_params_per_layer = available_per_layer - router_size

            if expert_params_per_layer > 0:
                expert_size = int(expert_params_per_layer / (3 * hidden_size * NUM_ACTIVE_EXPERTS))
                expert_size = max(8, min(expert_size, analysis['intermediate_size']))
            else:
                expert_size = 8  # Minimum viable

            # Compress each projection
            gate_compressed, _ = svd_compress_cpu(gate_orig, expert_size)
            up_compressed, _ = svd_compress_cpu(up_orig, expert_size)
            down_compressed, _ = svd_compress_cpu(down_orig, expert_size)

            # Create experts
            gate_experts = gate_compressed.unsqueeze(0).expand(NUM_TOTAL_EXPERTS, -1, -1).clone()
            up_experts = up_compressed.unsqueeze(0).expand(NUM_TOTAL_EXPERTS, -1, -1).clone()
            down_experts = down_compressed.unsqueeze(0).expand(NUM_TOTAL_EXPERTS, -1, -1).clone()

            # Router
            router_weight = torch.empty(hidden_size, NUM_TOTAL_EXPERTS)
            nn.init.uniform_(router_weight, -0.05, 0.05)
            router_weight[:, 0] += 0.1

            new_layer_state[mlp_prefix + "gate_experts"] = gate_experts
            new_layer_state[mlp_prefix + "up_experts"] = up_experts
            new_layer_state[mlp_prefix + "down_experts"] = down_experts
            new_layer_state[mlp_prefix + "router.weight"] = router_weight

            del gate_orig, up_orig, down_orig
            del gate_compressed, up_compressed, down_compressed
            gc.collect()
    else:
        # Regular MLP compression
        for proj_name in ["gate_proj", "up_proj", "down_proj"]:
            key = mlp_prefix + proj_name + ".weight"
            if key in layer_state:
                W = layer_state[key]
                W_compressed, _ = svd_compress_cpu(W, budget['new_intermediate_size'])
                new_layer_state[key] = W_compressed
                del W, W_compressed

                bias_key = mlp_prefix + proj_name + ".bias"
                if bias_key in layer_state:
                    new_layer_state[bias_key] = layer_state[bias_key]

    # Copy LayerNorm
    for norm_name in ["input_layernorm.weight", "post_attention_layernorm.weight"]:
        key = prefix + norm_name
        if key in layer_state:
            new_layer_state[key] = layer_state[key]

    for bias_name in ["input_layernorm.bias", "post_attention_layernorm.bias"]:
        key = prefix + bias_name
        if key in layer_state:
            new_layer_state[key] = layer_state[bias_name]

    return new_layer_state


def main():
    print("=" * 80)
    print("Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling")
    print("=" * 80)

    # Initialize managers
    memory_manager = MemoryManager(use_gpu=USE_GPU, gpu_memory_fraction=GPU_MEMORY_FRACTION)
    layer_storage = LayerStorage(TEMP_DIR) if SAVE_LAYERS_TO_DISK else None

    # Load config
    print(f"\nLoading configuration from: {MODEL_ID}")
    config = AutoConfig.from_pretrained(MODEL_ID)

    # Analyze model
    analysis = analyze_model(config)
    print(f"\n--- Model Analysis ---")
    print(f"Total params: {analysis['total_params']:,}")
    print(f"Target params: {TARGET_ACTIVE_PARAMS:,}")

    # Calculate budget
    budget = calculate_optimal_budget(
        analysis, TARGET_ACTIVE_PARAMS, ALLOW_EMBEDDING_DOWNSIZING,
        DOWNSIZE_ATTENTION or not DOWNSIZE_MLP_ONLY
    )

    # Load model weights
    print(f"\nLoading model weights...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float32, device_map="cpu", low_cpu_mem_usage=True
    )
    state_dict = model.state_dict()
    del model
    gc.collect()

    # Process embedding
    final_state_dict = process_embedding(state_dict, budget['new_hidden_size'], analysis, memory_manager)

    # Remove embedding from state_dict to free memory
    for key in ["model.embed_tokens.weight", "lm_head.weight"]:
        if key in state_dict:
            del state_dict[key]
    gc.collect()

    # Process each layer
    print("\n--- Processing Transformer Layers ---")

    for layer_idx in range(analysis['num_layers']):
        print(f"\nLayer {layer_idx + 1}/{analysis['num_layers']}")

        # Extract current layer
        prefix = f"model.layers.{layer_idx}."
        layer_state = {}
        keys_to_remove = []

        for key in state_dict.keys():
            if key.startswith(prefix):
                layer_state[key] = state_dict[key]
                keys_to_remove.append(key)

        # Remove layer from state_dict to free memory
        for key in keys_to_remove:
            del state_dict[key]
        gc.collect()

        # Process layer
        processed_layer = process_single_layer(layer_idx, layer_state, analysis, budget, memory_manager)

        # Save processed layer to disk
        if SAVE_LAYERS_TO_DISK:
            layer_storage.save_layer(layer_idx, processed_layer)
            del processed_layer
        else:
            final_state_dict.update(processed_layer)

        # Clean up
        del layer_state
        gc.collect()

        # Memory status
        mem_info = memory_manager.get_memory_info()
        print(f"  CPU Memory: {mem_info['cpu_percent']:.1f}% used")

        # Force cleanup if memory is high
        if mem_info['cpu_percent'] > MAX_CPU_MEMORY_PERCENT:
            print(f"  WARNING: High memory usage, forcing garbage collection...")
            gc.collect()
            if memory_manager.use_gpu:
                memory_manager.clear_gpu_cache()

    # Copy final layernorm
    for key in state_dict:
        if "model.norm" in key:
            final_state_dict[key] = state_dict[key]

    # Free state_dict
    del state_dict
    gc.collect()

    # Load layers from disk if needed
    if SAVE_LAYERS_TO_DISK:
        print("\n--- Merging Layers from Disk ---")
        for layer_idx in range(analysis['num_layers']):
            layer_state = layer_storage.load_layer(layer_idx)
            if layer_state:
                # Remove layer prefix from keys
                for key, value in layer_state.items():
                    final_state_dict[key] = value
                del layer_state
                gc.collect()

            # Clean up disk file
            layer_storage.cleanup_layer(layer_idx)

    # Calculate final params
    print("\n--- Summary ---")
    total_params = sum(t.numel() for t in final_state_dict.values())
    print(f"Total parameters: {total_params:,}")
    print(f"Target: {TARGET_ACTIVE_PARAMS:,}")
    print(f"Compression ratio: {analysis['total_params'] / total_params:.2f}x")

    # Update config
    updated_config = config
    if budget['new_hidden_size'] != analysis['hidden_size']:
        updated_config.hidden_size = analysis['hidden_size']
        updated_config.embedding_size = budget['new_hidden_size']
        updated_config.has_embed_proj = True

    if CONVERT_TO_MOE:
        updated_config.moe_num_experts = NUM_TOTAL_EXPERTS
        updated_config.moe_active_experts = NUM_ACTIVE_EXPERTS
        # Find expert size from first layer
        first_layer_key = f"model.layers.0.mlp.gate_experts"
        if first_layer_key in final_state_dict:
            updated_config.moe_expert_size = final_state_dict[first_layer_key].shape[1]
            updated_config.intermediate_size = updated_config.moe_expert_size
    else:
        updated_config.intermediate_size = budget['new_intermediate_size']

    # Save model
    print(f"\nSaving model to: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Save in chunks to avoid memory issues
    print("Saving weights in chunks...")
    temp_save_path = os.path.join(OUTPUT_DIR, "temp_save")
    os.makedirs(temp_save_path, exist_ok=True)

    # Save layer by layer
    chunk_size = 4  # Save 4 layers at a time
    all_keys = list(final_state_dict.keys())
    chunk_dict = {}

    for i, key in enumerate(all_keys):
        chunk_dict[key] = final_state_dict[key]

        if (i + 1) % chunk_size == 0 or i == len(all_keys) - 1:
            chunk_file = os.path.join(temp_save_path, f"chunk_{i // chunk_size:03d}.pt")
            torch.save(chunk_dict, chunk_file)
            del chunk_dict
            chunk_dict = {}
            gc.collect()

    # Merge chunks into final safetensors
    print("Merging chunks...")
    merged_state = {}
    chunk_files = sorted([f for f in os.listdir(temp_save_path) if f.endswith('.pt')])

    for chunk_file in chunk_files:
        chunk_path = os.path.join(temp_save_path, chunk_file)
        chunk_data = torch.load(chunk_path, map_location='cpu')
        merged_state.update(chunk_data)
        del chunk_data
        gc.collect()

    # Save final
    save_file(merged_state, os.path.join(OUTPUT_DIR, "model.safetensors"))

    # Clean up temp files
    shutil.rmtree(temp_save_path)
    if layer_storage:
        layer_storage.cleanup_all()

    # Save config
    print("Saving config...")
    updated_config.save_pretrained(OUTPUT_DIR)

    # Save tokenizer
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print("Tokenizer saved.")
    except:
        print("Could not save tokenizer.")

    print("\n✅ Model surgery complete!")
    print(f"Output saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling
Using GPU: Tesla T4
Total GPU memory: 14.56 GB

Loading configuration from: openbmb/MiniCPM5-1B-SFT



--- Model Analysis ---
Total params: 1,137,180,672
Target params: 500,000,000.0

--- Budget Allocation ---
Required reduction ratio: 0.4397
Reducing embedding hidden size to 765
Embedding + LM Head params: 199,756,800
Remaining budget: 300,243,200.0
Attention budget: 120,097,280.0
MLP budget: 180,145,920.0
New attention rank: 1118
New intermediate size: 1628

Loading model weights...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]


--- Processing Embedding & LM Head ---
Compressing embedding...
Compressing lm_head...

--- Processing Transformer Layers ---

Layer 1/24
  CPU Memory: 67.6% used

Layer 2/24
  CPU Memory: 66.4% used

Layer 3/24
  CPU Memory: 65.5% used

Layer 4/24
  CPU Memory: 64.7% used

Layer 5/24
  CPU Memory: 64.0% used

Layer 6/24
  CPU Memory: 63.1% used

Layer 7/24
  CPU Memory: 62.3% used

Layer 8/24
  CPU Memory: 61.5% used

Layer 9/24
  CPU Memory: 60.7% used

Layer 10/24
  CPU Memory: 60.0% used

Layer 11/24
  CPU Memory: 59.0% used

Layer 12/24
  CPU Memory: 58.3% used

Layer 13/24
  CPU Memory: 57.4% used

Layer 14/24
  CPU Memory: 56.6% used

Layer 15/24
  CPU Memory: 55.9% used

Layer 16/24
  CPU Memory: 54.9% used

Layer 17/24
  CPU Memory: 54.2% used

Layer 18/24
  CPU Memory: 53.2% used

Layer 19/24
  CPU Memory: 52.6% used

Layer 20/24
  CPU Memory: 51.9% used

Layer 21/24
  CPU Memory: 50.8% used

Layer 22/24
  CPU Memory: 50.1% used

Layer 23/24
  CPU Memory: 49.3% used

Layer 2

In [ ]:
import shutil
from IPython.display import FileLink
import os

# The OUTPUT_DIR is defined in the previous cell
output_zip_name = OUTPUT_DIR.strip('./') # Remove leading ./ for cleaner zip file name
zip_filename = f"{output_zip_name}.zip"

print(f"Zipping the output model directory: {OUTPUT_DIR} to {zip_filename}...")
shutil.make_archive(output_zip_name, 'zip', OUTPUT_DIR)

print(f"\nDownload the zipped model here:")
FileLink(zip_filename)

In [ ]:
"""
Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling
Using Autoencoder-based Compression instead of SVD
---------------------------------------------------------------------------
Memory-optimized version that processes layers one at a time and saves
directly to disk to prevent OOM errors.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import json
import os
import gc
import shutil
import tempfile
import time
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import save_file, load_file
from safetensors import safe_open
import psutil
from collections import OrderedDict
import numpy as np

# ==========================================
# USER CONTROL & PARAMETER TWEAKING PANEL
# ==========================================
TARGET_ACTIVE_PARAMS = 5.0e8           # Target 500M active parameters
ALLOW_EMBEDDING_DOWNSIZING = True      # If True, downscales token embedding dim via Autoencoder
DOWNSIZE_ATTENTION = True              # Keep Attention weights intact, or squeeze them
DOWNSIZE_MLP_ONLY = False              # Concentrate all autoencoder latent squeezing strictly on MLP/FFN layers

# MoE Architectural Upcycling Settings
CONVERT_TO_MOE = True                  # Turn the squeezed MLP layer into an MoE layout
NUM_TOTAL_EXPERTS = 4                  # Reduced from 8 to save memory
NUM_ACTIVE_EXPERTS = 2                 # Top-K active routing choice per token

# Autoencoder Training Settings
AE_EPOCHS = 50                         # Training epochs for autoencoder
AE_BATCH_SIZE = 32                     # Batch size for autoencoder training
AE_LEARNING_RATE = 1e-4               # Learning rate for autoencoder
AE_USE_GPU = True                     # Use GPU for autoencoder training
AE_VALIDATION_SPLIT = 0.2             # Validation split ratio

# Memory Management Settings
USE_GPU = True                         # Use GPU if available
GPU_MEMORY_FRACTION = 0.8              # Maximum fraction of GPU memory to use
SAVE_LAYERS_TO_DISK = True             # Save each layer to disk immediately
TEMP_DIR = "./temp_layer_storage"      # Directory for temporary layer files
MAX_CPU_MEMORY_PERCENT = 70            # Maximum CPU memory usage before forcing disk save

# Model source
MODEL_ID = "openbmb/MiniCPM5-1B-SFT"

# Output directory
OUTPUT_DIR = "./downsized_moe_model_ae"
# ==========================================


class Autoencoder(nn.Module):
    """Simple autoencoder for matrix compression."""

    def __init__(self, input_dim: int, latent_dim: int, hidden_dim: Optional[int] = None):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim or max(input_dim * 2, latent_dim * 4)

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, latent_dim),
            nn.Tanh()  # Bound latent representation
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def get_compression_ratio(self):
        """Calculate compression ratio."""
        encoder_params = sum(p.numel() for p in self.encoder.parameters())
        decoder_params = sum(p.numel() for p in self.decoder.parameters())
        total_params = encoder_params + decoder_params
        original_params = self.input_dim * self.input_dim
        return original_params / total_params


class LayerAutoencoder(nn.Module):
    """Autoencoder for compressing a specific layer matrix."""

    def __init__(self, original_matrix: torch.Tensor, latent_dim: int,
                 hidden_dim: Optional[int] = None):
        super().__init__()
        self.original_matrix = original_matrix.detach().clone()
        self.m, self.n = original_matrix.shape
        self.latent_dim = latent_dim

        # Create autoencoder
        self.autoencoder = Autoencoder(self.n, latent_dim, hidden_dim)

        # Store compressed representation
        self.compressed_matrix = None
        self.encoder_weights = None
        self.decoder_weights = None

    def fit(self, epochs: int = 50, batch_size: int = 32,
            learning_rate: float = 1e-4, use_gpu: bool = True,
            validation_split: float = 0.2, verbose: bool = True):
        """Train the autoencoder on the matrix data."""
        device = torch.device("cuda" if use_gpu and torch.cuda.is_available() else "cpu")
        self.autoencoder.to(device)

        # Prepare data - treat each column as a sample
        data = self.original_matrix.T  # Shape: (n, m)
        data = data.float().to(device)

        # Split data
        n_samples = data.shape[0]
        n_train = int(n_samples * (1 - validation_split))
        indices = torch.randperm(n_samples)
        train_indices = indices[:n_train]
        val_indices = indices[n_train:]

        train_data = data[train_indices]
        val_data = data[val_indices]

        # Optimizer
        optimizer = torch.optim.Adam(self.autoencoder.parameters(), lr=learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )

        # Training loop
        best_val_loss = float('inf')
        best_state_dict = None

        if verbose:
            print(f"  Training autoencoder: {self.m}x{self.n} -> latent={self.latent_dim}")
            print(f"  Training samples: {n_train}, Validation samples: {len(val_indices)}")

        for epoch in range(epochs):
            # Training
            self.autoencoder.train()
            train_loss = 0.0
            for i in range(0, n_train, batch_size):
                batch = train_data[i:min(i+batch_size, n_train)]
                optimizer.zero_grad()

                reconstructed, _ = self.autoencoder(batch)
                loss = F.mse_loss(reconstructed, batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * batch.size(0)

            train_loss /= n_train

            # Validation
            self.autoencoder.eval()
            val_loss = 0.0
            with torch.no_grad():
                for i in range(0, len(val_data), batch_size):
                    batch = val_data[i:min(i+batch_size, len(val_data))]
                    reconstructed, _ = self.autoencoder(batch)
                    loss = F.mse_loss(reconstructed, batch)
                    val_loss += loss.item() * batch.size(0)

            val_loss /= len(val_data) if len(val_data) > 0 else 1

            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = {k: v.cpu().clone() for k, v in self.autoencoder.state_dict().items()}

            if verbose and (epoch + 1) % 10 == 0:
                print(f"    Epoch {epoch+1}/{epochs}: Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

        # Load best model
        if best_state_dict is not None:
            self.autoencoder.load_state_dict(best_state_dict, strict=False)

        # Get compressed representation
        self.autoencoder.eval()
        with torch.no_grad():
            self.autoencoder.to('cpu')
            data_cpu = data.cpu()
            _, encoded = self.autoencoder(data_cpu)

            # Store compression matrices
            self.encoder_weights = self.autoencoder.encoder[0].weight.data.cpu().clone()  # First linear layer
            self.decoder_weights = self.autoencoder.decoder[2].weight.data.cpu().clone()  # Last linear layer

            # Reconstruct compressed matrix
            reconstructed = self.autoencoder.decoder(encoded)
            self.compressed_matrix = reconstructed.T  # Back to original shape (m, n)

        # Clean up
        self.autoencoder.to('cpu')
        del data, train_data, val_data
        gc.collect()

        return best_val_loss

    def compress(self) -> torch.Tensor:
        """Return the compressed matrix."""
        if self.compressed_matrix is None:
            raise ValueError("Autoencoder not trained yet. Call fit() first.")
        return self.compressed_matrix

    def get_compression_weights(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return encoder and decoder weights for reconstruction."""
        if self.encoder_weights is None or self.decoder_weights is None:
            raise ValueError("Autoencoder not trained yet. Call fit() first.")
        return self.encoder_weights, self.decoder_weights

    def save(self, path: str):
        """Save autoencoder state."""
        torch.save({
            'autoencoder_state': self.autoencoder.state_dict(),
            'compressed_matrix': self.compressed_matrix,
            'encoder_weights': self.encoder_weights,
            'decoder_weights': self.decoder_weights,
            'm': self.m,
            'n': self.n,
            'latent_dim': self.latent_dim
        }, path)

    @classmethod
    def load(cls, path: str):
        """Load autoencoder from file."""
        data = torch.load(path, map_location='cpu')
        instance = cls.__new__(cls)
        instance.autoencoder = Autoencoder(data['n'], data['latent_dim'])
        instance.autoencoder.load_state_dict(data['autoencoder_state'])
        instance.compressed_matrix = data['compressed_matrix']
        instance.encoder_weights = data['encoder_weights']
        instance.decoder_weights = data['decoder_weights']
        instance.m = data['m']
        instance.n = data['n']
        instance.latent_dim = data['latent_dim']
        return instance


class MemoryManager:
    """Manages GPU/CPU memory during model processing."""

    def __init__(self, use_gpu: bool = True, gpu_memory_fraction: float = 0.8):
        self.use_gpu = use_gpu and torch.cuda.is_available()
        self.gpu_memory_fraction = gpu_memory_fraction
        self.device = torch.device("cuda" if self.use_gpu else "cpu")

        if self.use_gpu:
            torch.cuda.set_per_process_memory_fraction(gpu_memory_fraction)
            self.gpu_name = torch.cuda.get_device_name(0)
            self.total_gpu_memory = torch.cuda.get_device_properties(0).total_memory
            print(f"Using GPU: {self.gpu_name}")
            print(f"Total GPU memory: {self.total_gpu_memory / 1024**3:.2f} GB")

    def get_memory_info(self) -> Dict[str, float]:
        """Get current memory usage."""
        info = {}
        if self.use_gpu:
            info['gpu_allocated'] = torch.cuda.memory_allocated() / 1024**3
            info['gpu_cached'] = torch.cuda.memory_reserved() / 1024**3
            info['gpu_free'] = (self.total_gpu_memory - torch.cuda.memory_allocated()) / 1024**3

        cpu_memory = psutil.virtual_memory()
        info['cpu_used'] = cpu_memory.used / 1024**3
        info['cpu_available'] = cpu_memory.available / 1024**3
        info['cpu_percent'] = cpu_memory.percent

        return info

    def clear_gpu_cache(self):
        """Clear GPU cache."""
        if self.use_gpu:
            torch.cuda.empty_cache()
            gc.collect()

    def should_use_gpu(self, tensor_size_gb: float) -> bool:
        """Check if tensor fits in GPU memory."""
        if not self.use_gpu:
            return False
        free_gpu_gb = (self.total_gpu_memory - torch.cuda.memory_allocated()) / 1024**3
        return tensor_size_gb * 3 < free_gpu_gb * 0.8  # Autoencoder needs ~3x memory


class LayerStorage:
    """Manages saving/loading layers to disk to save RAM."""

    def __init__(self, temp_dir: str = "./temp_layer_storage"):
        self.temp_dir = temp_dir
        os.makedirs(temp_dir, exist_ok=True)
        self.layer_files = {}

    def save_layer(self, layer_idx: int, layer_state: Dict[str, torch.Tensor]):
        """Save a layer's state dict to disk."""
        filename = os.path.join(self.temp_dir, f"layer_{layer_idx:03d}.pt")
        torch.save(layer_state, filename)
        self.layer_files[layer_idx] = filename
        return filename

    def load_layer(self, layer_idx: int) -> Dict[str, torch.Tensor]:
        """Load a layer's state dict from disk."""
        filename = self.layer_files.get(layer_idx)
        if filename and os.path.exists(filename):
            return torch.load(filename, map_location='cpu')
        return None

    def cleanup_layer(self, layer_idx: int):
        """Remove a layer's file from disk."""
        filename = self.layer_files.get(layer_idx)
        if filename and os.path.exists(filename):
            os.remove(filename)
            del self.layer_files[layer_idx]

    def cleanup_all(self):
        """Remove all temporary files."""
        if os.path.exists(self.temp_dir):
            shutil.rmtree(self.temp_dir)


def autoencoder_compress(W: torch.Tensor, target_rank: int,
                         memory_manager: MemoryManager,
                         epochs: int = AE_EPOCHS,
                         batch_size: int = AE_BATCH_SIZE,
                         learning_rate: float = AE_LEARNING_RATE,
                         verbose: bool = True) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
    """Compress matrix using autoencoder."""

    # Move to CPU for autoencoder training
    W_cpu = W.float().cpu()

    # Determine if we should use GPU for training
    use_gpu = AE_USE_GPU and memory_manager.use_gpu
    if use_gpu:
        # Check if matrix fits in GPU memory
        matrix_size_gb = W_cpu.numel() * 4 / 1024**3  # float32
        if not memory_manager.should_use_gpu(matrix_size_gb * 2):  # Need double for training
            use_gpu = False
            print("  Matrix too large for GPU, using CPU for autoencoder training")

    # Handle very large matrices with randomized sampling
    if W_cpu.shape[0] * W_cpu.shape[1] > 50_000_000:  # >50M elements
        print("  Large matrix detected, using randomized subset for training...")
        # Sample rows and columns for training
        n_samples = min(5000, W_cpu.shape[0])
        sample_indices = torch.randperm(W_cpu.shape[0])[:n_samples]
        W_sample = W_cpu[sample_indices].T  # Use sampled rows as training data

        # Train autoencoder on sampled data
        ae = LayerAutoencoder(W_sample, target_rank)
        ae.fit(epochs=epochs, batch_size=batch_size,
               learning_rate=learning_rate, use_gpu=use_gpu,
               validation_split=AE_VALIDATION_SPLIT, verbose=verbose)

        # Apply to full matrix
        compressed = ae.compress()
        # Expand back to full size
        full_compressed = torch.zeros_like(W_cpu)
        full_compressed[sample_indices] = compressed
        # Fill remaining rows with average
        remaining_indices = [i for i in range(W_cpu.shape[0]) if i not in sample_indices]
        if remaining_indices:
            avg_row = compressed.mean(dim=0, keepdim=True)
            for idx in remaining_indices:
                full_compressed[idx] = avg_row

        encoder_weights, decoder_weights = ae.get_compression_weights()
    else:
        # Train autoencoder on full matrix
        ae = LayerAutoencoder(W_cpu, target_rank)
        ae.fit(epochs=epochs, batch_size=batch_size,
               learning_rate=learning_rate, use_gpu=use_gpu,
               validation_split=AE_VALIDATION_SPLIT, verbose=verbose)

        full_compressed = ae.compress()
        encoder_weights, decoder_weights = ae.get_compression_weights()

    # Clean up
    del W_cpu
    gc.collect()
    if memory_manager.use_gpu:
        memory_manager.clear_gpu_cache()

    return full_compressed.to(W.dtype), (encoder_weights, decoder_weights)


def analyze_model(config: AutoConfig) -> Dict[str, Any]:
    """Analyze model dimensions and parameter counts."""
    hidden_size = config.hidden_size
    vocab_size = config.vocab_size
    num_layers = config.num_hidden_layers
    intermediate_size = config.intermediate_size

    embedding_params = vocab_size * hidden_size
    lm_head_params = vocab_size * hidden_size
    attn_params_per_layer = 4 * hidden_size * hidden_size
    mlp_params_per_layer = 3 * hidden_size * intermediate_size

    total_attention = num_layers * attn_params_per_layer
    total_mlp = num_layers * mlp_params_per_layer
    total = embedding_params + lm_head_params + total_attention + total_mlp

    return {
        "hidden_size": hidden_size,
        "vocab_size": vocab_size,
        "num_layers": num_layers,
        "intermediate_size": intermediate_size,
        "embedding_params": embedding_params,
        "lm_head_params": lm_head_params,
        "attention_params": total_attention,
        "mlp_params": total_mlp,
        "total_params": total,
        "attn_params_per_layer": attn_params_per_layer,
        "mlp_params_per_layer": mlp_params_per_layer,
    }


def calculate_optimal_budget(analysis, target_params, allow_embedding_downsize, downsample_attention):
    """Calculate optimal budget allocation."""
    hidden_size = analysis['hidden_size']
    num_layers = analysis['num_layers']
    total_params = analysis['total_params']
    reduction_ratio = target_params / total_params

    print(f"\n--- Budget Allocation ---")
    print(f"Required reduction ratio: {reduction_ratio:.4f}")

    fixed_params = analysis['embedding_params'] + analysis['lm_head_params']

    new_hidden_size = hidden_size
    if allow_embedding_downsize and fixed_params > target_params * 0.5:
        remaining_for_embedding = target_params * 0.4
        vocab_size = analysis['vocab_size']
        new_hidden_size = int(remaining_for_embedding / (2 * vocab_size))
        new_hidden_size = max(64, min(new_hidden_size, hidden_size))
        print(f"Reducing embedding hidden size to {new_hidden_size}")

    embedding_lm_head_params = analysis['vocab_size'] * new_hidden_size * 2
    remaining_budget = target_params - embedding_lm_head_params

    if downsample_attention:
        attention_budget = remaining_budget * 0.4
        mlp_budget = remaining_budget * 0.6
    else:
        attention_budget = analysis['attention_params']
        mlp_budget = remaining_budget - attention_budget
        if mlp_budget < 0:
            downsample_attention = True
            attention_budget = remaining_budget * 0.4
            mlp_budget = remaining_budget * 0.6

    attn_reduction = attention_budget / analysis['attention_params'] if analysis['attention_params'] > 0 else 1.0
    new_attn_rank = max(1, int(hidden_size * math.sqrt(max(attn_reduction, 0.01))))

    if mlp_budget > 0:
        new_intermediate_size = int(mlp_budget / (3 * hidden_size * num_layers))
    else:
        new_intermediate_size = 1

    new_intermediate_size = max(8, min(new_intermediate_size, analysis['intermediate_size']))
    new_attn_rank = max(8, min(new_attn_rank, hidden_size))

    print(f"Embedding + LM Head params: {embedding_lm_head_params:,}")
    print(f"Remaining budget: {remaining_budget:,}")
    print(f"Attention budget: {attention_budget:,}")
    print(f"MLP budget: {mlp_budget:,}")
    print(f"New attention rank: {new_attn_rank}")
    print(f"New intermediate size: {new_intermediate_size}")

    return {
        "new_hidden_size": new_hidden_size,
        "new_intermediate_size": new_intermediate_size,
        "new_attn_rank": new_attn_rank,
        "downsample_attention": downsample_attention,
        "attention_budget": attention_budget,
        "mlp_budget": mlp_budget,
    }


def process_embedding(state_dict, new_hidden_size, analysis, memory_manager):
    """Process embedding and lm_head with autoencoder compression."""
    print("\n--- Processing Embedding & LM Head ---")

    embed_key = "model.embed_tokens.weight"
    lm_head_key = "lm_head.weight"
    new_state = {}

    if new_hidden_size != analysis['hidden_size']:
        print("Compressing embedding with autoencoder...")
        embed_weight = state_dict[embed_key]
        embed_compressed, (encoder_w, decoder_w) = autoencoder_compress(
            embed_weight.T, new_hidden_size, memory_manager, epochs=30  # Fewer epochs for embedding
        )
        new_state[embed_key] = embed_compressed.T

        # Store compression matrices for reconstruction
        new_state["model.embed_encoder.weight"] = encoder_w
        new_state["model.embed_decoder.weight"] = decoder_w

        print("Compressing lm_head with autoencoder...")
        if lm_head_key in state_dict:
            lm_head_weight = state_dict[lm_head_key]
            lm_head_compressed, (enc_w, dec_w) = autoencoder_compress(
                lm_head_weight.T, new_hidden_size, memory_manager, epochs=30
            )
            new_state[lm_head_key] = lm_head_compressed.T
            new_state["model.lm_head_encoder.weight"] = enc_w
            new_state["model.lm_head_decoder.weight"] = dec_w
        else:
            new_state[lm_head_key] = new_state[embed_key].clone()
            new_state["model.lm_head_encoder.weight"] = encoder_w.clone()
            new_state["model.lm_head_decoder.weight"] = decoder_w.clone()

        del embed_weight, embed_compressed
        if 'lm_head_weight' in locals():
            del lm_head_weight
        if 'lm_head_compressed' in locals():
            del lm_head_compressed
        gc.collect()
    else:
        new_state[embed_key] = state_dict[embed_key]
        if lm_head_key in state_dict:
            new_state[lm_head_key] = state_dict[lm_head_key]
        else:
            new_state[lm_head_key] = new_state[embed_key].clone()

    return new_state


def process_single_layer(layer_idx, layer_state, analysis, budget, memory_manager):
    """Process a single transformer layer with autoencoder compression."""
    prefix = f"model.layers.{layer_idx}."
    new_layer_state = {}

    hidden_size = analysis['hidden_size']
    new_attn_rank = budget['new_attn_rank']

    # Process attention with autoencoder
    attn_prefix = prefix + "self_attn."
    for proj_name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        key = attn_prefix + proj_name + ".weight"
        if key in layer_state:
            if budget['downsample_attention']:
                W = layer_state[key]
                print(f"  Compressing {proj_name} with autoencoder...")
                W_compressed, (enc_w, dec_w) = autoencoder_compress(
                    W, new_attn_rank, memory_manager
                )
                new_layer_state[key] = W_compressed
                # Store compression info
                new_layer_state[f"{key}_encoder"] = enc_w
                new_layer_state[f"{key}_decoder"] = dec_w
                del W, W_compressed
            else:
                new_layer_state[key] = layer_state[key]

            bias_key = attn_prefix + proj_name + ".bias"
            if bias_key in layer_state:
                new_layer_state[bias_key] = layer_state[bias_key]

    # Process MLP with autoencoder
    mlp_prefix = prefix + "mlp."

    if CONVERT_TO_MOE:
        gate_key = mlp_prefix + "gate_proj.weight"
        up_key = mlp_prefix + "up_proj.weight"
        down_key = mlp_prefix + "down_proj.weight"

        if gate_key in layer_state:
            gate_orig = layer_state[gate_key]
            up_orig = layer_state[up_key]
            down_orig = layer_state[down_key]

            # Calculate expert size
            available_per_layer = budget['mlp_budget'] / analysis['num_layers']
            router_size = hidden_size * NUM_TOTAL_EXPERTS
            expert_params_per_layer = available_per_layer - router_size

            if expert_params_per_layer > 0:
                expert_size = int(expert_params_per_layer / (3 * hidden_size * NUM_ACTIVE_EXPERTS))
                expert_size = max(8, min(expert_size, analysis['intermediate_size']))
            else:
                expert_size = 8  # Minimum viable

            # Compress each projection with autoencoder
            print(f"  Compressing MLP projections with autoencoder...")
            gate_compressed, (gate_enc, gate_dec) = autoencoder_compress(
                gate_orig, expert_size, memory_manager
            )
            up_compressed, (up_enc, up_dec) = autoencoder_compress(
                up_orig, expert_size, memory_manager
            )
            down_compressed, (down_enc, down_dec) = autoencoder_compress(
                down_orig, expert_size, memory_manager
            )

            # Create experts
            gate_experts = gate_compressed.unsqueeze(0).expand(NUM_TOTAL_EXPERTS, -1, -1).clone()
            up_experts = up_compressed.unsqueeze(0).expand(NUM_TOTAL_EXPERTS, -1, -1).clone()
            down_experts = down_compressed.unsqueeze(0).expand(NUM_TOTAL_EXPERTS, -1, -1).clone()

            # Router
            router_weight = torch.empty(hidden_size, NUM_TOTAL_EXPERTS)
            nn.init.uniform_(router_weight, -0.05, 0.05)
            router_weight[:, 0] += 0.1

            new_layer_state[mlp_prefix + "gate_experts"] = gate_experts
            new_layer_state[mlp_prefix + "up_experts"] = up_experts
            new_layer_state[mlp_prefix + "down_experts"] = down_experts
            new_layer_state[mlp_prefix + "router.weight"] = router_weight

            # Store compression info
            new_layer_state[mlp_prefix + "gate_encoder"] = gate_enc
            new_layer_state[mlp_prefix + "gate_decoder"] = gate_dec
            new_layer_state[mlp_prefix + "up_encoder"] = up_enc
            new_layer_state[mlp_prefix + "up_decoder"] = up_dec
            new_layer_state[mlp_prefix + "down_encoder"] = down_enc
            new_layer_state[mlp_prefix + "down_decoder"] = down_dec

            del gate_orig, up_orig, down_orig
            del gate_compressed, up_compressed, down_compressed
            gc.collect()
    else:
        # Regular MLP compression with autoencoder
        for proj_name in ["gate_proj", "up_proj", "down_proj"]:
            key = mlp_prefix + proj_name + ".weight"
            if key in layer_state:
                W = layer_state[key]
                print(f"  Compressing {proj_name} with autoencoder...")
                W_compressed, (enc_w, dec_w) = autoencoder_compress(
                    W, budget['new_intermediate_size'], memory_manager
                )
                new_layer_state[key] = W_compressed
                new_layer_state[f"{key}_encoder"] = enc_w
                new_layer_state[f"{key}_decoder"] = dec_w
                del W, W_compressed

                bias_key = mlp_prefix + proj_name + ".bias"
                if bias_key in layer_state:
                    new_layer_state[bias_key] = layer_state[bias_key]

    # Copy LayerNorm
    for norm_name in ["input_layernorm.weight", "post_attention_layernorm.weight"]:
        key = prefix + norm_name
        if key in layer_state:
            new_layer_state[key] = layer_state[key]

    for bias_name in ["input_layernorm.bias", "post_attention_layernorm.bias"]:
        key = prefix + bias_name
        if key in layer_state:
            new_layer_state[key] = layer_state[bias_name]

    return new_layer_state


def main():
    print("=" * 80)
    print("Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling")
    print("Using Autoencoder-based Compression")
    print("=" * 80)

    # Initialize managers
    memory_manager = MemoryManager(use_gpu=USE_GPU, gpu_memory_fraction=GPU_MEMORY_FRACTION)
    layer_storage = LayerStorage(TEMP_DIR) if SAVE_LAYERS_TO_DISK else None

    # Load config
    print(f"\nLoading configuration from: {MODEL_ID}")
    config = AutoConfig.from_pretrained(MODEL_ID)

    # Analyze model
    analysis = analyze_model(config)
    print(f"\n--- Model Analysis ---")
    print(f"Total params: {analysis['total_params']:,}")
    print(f"Target params: {TARGET_ACTIVE_PARAMS:,}")

    # Calculate budget
    budget = calculate_optimal_budget(
        analysis, TARGET_ACTIVE_PARAMS, ALLOW_EMBEDDING_DOWNSIZING,
        DOWNSIZE_ATTENTION or not DOWNSIZE_MLP_ONLY
    )

    # Load model weights
    print(f"\nLoading model weights...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float32, device_map="cpu", low_cpu_mem_usage=True
    )
    state_dict = model.state_dict()
    del model
    gc.collect()

    # Process embedding
    final_state_dict = process_embedding(state_dict, budget['new_hidden_size'], analysis, memory_manager)

    # Remove embedding from state_dict to free memory
    for key in ["model.embed_tokens.weight", "lm_head.weight"]:
        if key in state_dict:
            del state_dict[key]
    gc.collect()

    # Process each layer
    print("\n--- Processing Transformer Layers ---")

    for layer_idx in range(analysis['num_layers']):
        print(f"\nLayer {layer_idx + 1}/{analysis['num_layers']}")

        # Extract current layer
        prefix = f"model.layers.{layer_idx}."
        layer_state = {}
        keys_to_remove = []

        for key in state_dict.keys():
            if key.startswith(prefix):
                layer_state[key] = state_dict[key]
                keys_to_remove.append(key)

        # Remove layer from state_dict to free memory
        for key in keys_to_remove:
            del state_dict[key]
        gc.collect()

        # Process layer
        processed_layer = process_single_layer(layer_idx, layer_state, analysis, budget, memory_manager)

        # Save processed layer to disk
        if SAVE_LAYERS_TO_DISK:
            layer_storage.save_layer(layer_idx, processed_layer)
            del processed_layer
        else:
            final_state_dict.update(processed_layer)

        # Clean up
        del layer_state
        gc.collect()

        # Memory status
        mem_info = memory_manager.get_memory_info()
        print(f"  CPU Memory: {mem_info['cpu_percent']:.1f}% used")

        # Force cleanup if memory is high
        if mem_info['cpu_percent'] > MAX_CPU_MEMORY_PERCENT:
            print(f"  WARNING: High memory usage, forcing garbage collection...")
            gc.collect()
            if memory_manager.use_gpu:
                memory_manager.clear_gpu_cache()

    # Copy final layernorm
    for key in state_dict:
        if "model.norm" in key:
            final_state_dict[key] = state_dict[key]

    # Free state_dict
    del state_dict
    gc.collect()

    # Load layers from disk if needed
    if SAVE_LAYERS_TO_DISK:
        print("\n--- Merging Layers from Disk ---")
        for layer_idx in range(analysis['num_layers']):
            layer_state = layer_storage.load_layer(layer_idx)
            if layer_state:
                for key, value in layer_state.items():
                    final_state_dict[key] = value
                del layer_state
                gc.collect()

            # Clean up disk file
            layer_storage.cleanup_layer(layer_idx)

    # Calculate final params
    print("\n--- Summary ---")
    total_params = sum(t.numel() for t in final_state_dict.values())
    print(f"Total parameters: {total_params:,}")
    print(f"Target: {TARGET_ACTIVE_PARAMS:,}")
    print(f"Compression ratio: {analysis['total_params'] / total_params:.2f}x")

    # Update config
    updated_config = config
    if budget['new_hidden_size'] != analysis['hidden_size']:
        updated_config.hidden_size = analysis['hidden_size']
        updated_config.embedding_size = budget['new_hidden_size']
        updated_config.has_embed_proj = True

    if CONVERT_TO_MOE:
        updated_config.moe_num_experts = NUM_TOTAL_EXPERTS
        updated_config.moe_active_experts = NUM_ACTIVE_EXPERTS
        # Find expert size from first layer
        first_layer_key = f"model.layers.0.mlp.gate_experts"
        if first_layer_key in final_state_dict:
            updated_config.moe_expert_size = final_state_dict[first_layer_key].shape[1]
            updated_config.intermediate_size = updated_config.moe_expert_size
    else:
        updated_config.intermediate_size = budget['new_intermediate_size']

    # Save model
    print(f"\nSaving model to: {OUTPUT_DIR}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Save in chunks to avoid memory issues
    print("Saving weights in chunks...")
    temp_save_path = os.path.join(OUTPUT_DIR, "temp_save")
    os.makedirs(temp_save_path, exist_ok=True)

    # Save layer by layer
    chunk_size = 4  # Save 4 layers at a time
    all_keys = list(final_state_dict.keys())
    chunk_dict = {}

    for i, key in enumerate(all_keys):
        chunk_dict[key] = final_state_dict[key]

        if (i + 1) % chunk_size == 0 or i == len(all_keys) - 1:
            chunk_file = os.path.join(temp_save_path, f"chunk_{i // chunk_size:03d}.pt")
            torch.save(chunk_dict, chunk_file)
            del chunk_dict
            chunk_dict = {}
            gc.collect()

    # Merge chunks into final safetensors
    print("Merging chunks...")
    merged_state = {}
    chunk_files = sorted([f for f in os.listdir(temp_save_path) if f.endswith('.pt')])

    for chunk_file in chunk_files:
        chunk_path = os.path.join(temp_save_path, chunk_file)
        chunk_data = torch.load(chunk_path, map_location='cpu')
        merged_state.update(chunk_data)
        del chunk_data
        gc.collect()

    # Save final
    save_file(merged_state, os.path.join(OUTPUT_DIR, "model.safetensors"))

    # Clean up temp files
    shutil.rmtree(temp_save_path)
    if layer_storage:
        layer_storage.cleanup_all()

    # Save config
    print("Saving config...")
    updated_config.save_pretrained(OUTPUT_DIR)

    # Save tokenizer
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print("Tokenizer saved.")
    except:
        print("Could not save tokenizer.")

    print("\n✅ Model surgery complete with Autoencoder compression!")
    print(f"Output saved to: {OUTPUT_DIR}")

    # Print summary
    print("\n--- Compression Summary ---")
    print(f"Original parameters: {analysis['total_params']:,}")
    print(f"Compressed parameters: {total_params:,}")
    print(f"Compression ratio: {analysis['total_params'] / total_params:.2f}x")
    print(f"Reduction: {((analysis['total_params'] - total_params) / analysis['total_params'] * 100):.1f}%")


if __name__ == "__main__":
    main()

Parameter-Space Model Surgery: Zero-Shot Structural Truncation & Upcycling
Using Autoencoder-based Compression
Using GPU: Tesla T4
Total GPU memory: 14.56 GB

Loading configuration from: openbmb/MiniCPM5-1B-SFT



--- Model Analysis ---
Total params: 1,137,180,672
Target params: 500,000,000.0

--- Budget Allocation ---
Required reduction ratio: 0.4397
Reducing embedding hidden size to 765
Embedding + LM Head params: 199,756,800
Remaining budget: 300,243,200.0
Attention budget: 120,097,280.0
MLP budget: 180,145,920.0
New attention rank: 1118
New intermediate size: 1628

Loading model weights...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]


--- Processing Embedding & LM Head ---
Compressing embedding with autoencoder...
  Large matrix detected, using randomized subset for training...
  Training autoencoder: 130560x1536 -> latent=765
  Training samples: 1228, Validation samples: 308


RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x130560 and 1536x3072)